# Lab Ngày 26: Hạ Tầng MCP/A2A & Agentic Routing

**Khóa học:** AICB-P2T2 · Tuần 6 · Chương 6  
**Framework:** [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)  
**Thời lượng:** ~2 giờ

---

## Câu hỏi mở đầu

> *"Agent gọi agent — hạ tầng của bạn có scale được cho multi-agent choreography không?"*

Một hệ thống nghiên cứu cần 4 agent chuyên biệt phối hợp. Nếu không có chuẩn giao tiếp chung, mỗi agent nói "ngôn ngữ" riêng. **MCP + A2A** là giao thức phổ quát cho hệ sinh thái AI agent.

---

## Mục tiêu học tập

Sau buổi lab này, bạn sẽ:

1. Thiết kế và triển khai **hạ tầng MCP server** cho AI tools
2. Triển khai **giao tiếp A2A** giữa nhiều agent bằng ADK
3. Xây dựng **lớp agentic routing** với semantic routing
4. Áp dụng các mẫu **state, bảo mật và observability** trong hệ multi-agent

## Sản phẩm cuối ngày

| Thành phần | Yêu cầu |
|------------|--------|
| MCP server | 3 tools qua stdio (local) hoặc SSE (remote) |
| Agent registry | Health check + khám phá capability |
| Semantic router | Định tuyến request tới đúng specialist |
| Demo multi-agent | Orchestrator + 2 specialist qua A2A |
| Trace | Toàn bộ luồng multi-agent hiển thị trong log/trace |
| Data governance | Policy MCP/A2A + audit log + HITL gate |

---
## Module 0 — Cài đặt môi trường

Lab dùng **Conda** (môi trường khuyến nghị: `pii-env`). Trước khi chạy cell bên dưới:

```bash
conda create -n pii-env python=3.12 -y   # chỉ lần đầu
conda activate pii-env
```

Chọn kernel **pii-env** trong Jupyter nếu notebook hỏi.

Cài ADK kèm **extra A2A** và MCP SDK. Gói `google-adk` cơ bản **không** bao gồm dependency A2A.

In [1]:
# Chạy một lần sau khi conda activate pii-env (kernel Jupyter = pii-env)
!pip install -q -r requirements.txt

# Nếu vẫn thấy cảnh báo cryptography từ presidio cũ (2.2.360), nâng cấp:
# !pip install -q -U "presidio-anonymizer>=2.2.363"

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

# Google AI Studio (khuyến nghị cho lab)
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")

assert os.getenv("GOOGLE_API_KEY"), (
    "Đặt GOOGLE_API_KEY trong .env — lấy key tại https://aistudio.google.com/app/apikey"
)

print("✓ Môi trường sẵn sàng")
print(f"  Thư mục dự án: {PROJECT_ROOT}")

✓ Môi trường sẵn sàng
  Thư mục dự án: C:\Users\THAI NHI\Desktop\VinUni\Day26-Track02-Cohort2-MCP-A2A-Infrastructure


### Tổng quan kiến trúc

```
┌─────────────────────────────────────────────────────────────┐
│                     ORCHESTRATOR (ADK)                       │
│  Semantic Router → ủy quyền cho các specialist               │
└──────────┬──────────────────────────────┬───────────────────┘
           │ A2A (HTTP)                   │ MCP (stdio/SSE)
           ▼                              ▼
┌──────────────────┐            ┌──────────────────────┐
│  Search Agent    │            │  MCP Tools Server     │
│  :8001           │            │  search / sql / sum   │
├──────────────────┤            └──────────────────────┘
│  Database Agent  │
│  :8002           │
└──────────────────┘
```

**Điểm then chốt:** MCP chuẩn hóa truy cập *tool*; A2A chuẩn hóa giao tiếp *agent*. ADK kết nối cả hai.

---
## ⚠️ Bước bắt buộc — Khởi động A2A Specialist Servers

**Phải làm trước Module 2, Module 4 và Capstone.** Nếu bỏ qua, kiểm tra agent card sẽ báo `[CHƯA ĐẠT]`.

**Điều kiện:** đã cài dependency (Module 0) và **đang ở thư mục dự án**. Kích hoạt conda env:

```bash
conda activate pii-env
export PYTHONPATH=$PWD
```

### Cách 1 — Từ notebook (khuyến nghị)

Chạy cell bên dưới để khởi động cả hai server nền. Đợi thông báo `search OK` / `database OK`, rồi chạy cell kiểm tra.

### Cách 2 — Hai terminal riêng

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD

# Terminal 1
python -m uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8001

# Terminal 2
python -m uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8002
```

Hoặc: `bash scripts/start_search_agent.sh` và `bash scripts/start_database_agent.sh`

### Cách 3 — Một lệnh (nền)

```bash
bash scripts/start_a2a_servers.sh
```

### Kiểm tra / Dừng

```bash
curl http://localhost:8001/.well-known/agent-card.json
curl http://localhost:8002/.well-known/agent-card.json
bash scripts/stop_a2a_servers.sh
```

In [3]:
# Khởi động A2A servers nền (chạy cell này một lần mỗi phiên lab)
import subprocess
from pathlib import Path

script = PROJECT_ROOT / "scripts" / "start_a2a_servers.sh"
if script.exists():
    result = subprocess.run(["bash", str(script)], cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Không tìm thấy script. Chạy thủ công:")
    print("  uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8001")
    print("  uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8002")


/bin/bash: C:\Users\THAI NHI\Desktop\VinUni\Day26-Track02-Cohort2-MCP-A2A-Infrastructure\scripts\start_a2a_servers.sh: No such file or directory



In [4]:
# Kiểm tra agent card — phải thấy ✓ trước khi sang Module 4 / Capstone
import httpx

AGENT_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

all_ok = True
for name, url in AGENT_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        card = r.json()
        print(f"✓ {name}: {card.get('name')} — {str(card.get('description', ''))[:50]}...")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")
        port = {"search_agent": 8001, "database_agent": 8002, "synthesis_agent": 8003}[name]
        print(f"  → Chạy: uvicorn agents.{name}.agent:a2a_app --host localhost --port {port}")

if all_ok:
    print("\n✓ A2A servers sẵn sàng — tiếp tục lab")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

✓ search_agent: search_agent — Tìm kiếm web và trả về đoạn trích liên quan cho tá...
✓ database_agent: database_agent — Truy vấn database chỉ đọc và trả về metrics có cấu...
✓ synthesis_agent: synthesis_agent — Tổng hợp kết quả nghiên cứu thành báo cáo cuối có ...

✓ A2A servers sẵn sàng — tiếp tục lab


---
## Module 1 — Model Context Protocol (MCP)

### Lý thuyết (từ slide)

| Lớp | Vai trò |
|-----|--------|
| **Tools Server** | Thực thi hàm (search, SQL, API) |
| **Resources Server** | Expose dữ liệu (file, hàng DB) |
| **Prompts Server** | Template prompt tái sử dụng |

**Transport:** `stdio` (dev local), `HTTP+SSE` (remote/K8s), `WebSocket` (streaming hai chiều)

**Quy tắc thiết kế:**
- Docstring rõ ràng (LLM đọc để chọn tool)
- Type hint cho mọi tham số
- Validate input trước khi thực thi
- SQL tool mặc định chỉ đọc (read-only)
- Log mọi lần gọi tool

### 📝 Bài tập 1.1 — Khám phá MCP Server

Mở `mcp_server/research_tools_server.py` và trả lời:
1. Ba tool nào được expose?
2. `_sql_query` enforce governance như thế nào?
3. Vì sao dùng transport stdio khi dev local?

In [5]:
# Demo: gọi logic MCP tool trực tiếp (không cần khởi động server)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "mcp_server"))

from research_tools_server import _search_documents, _sql_query, _summarize_text

print("=== search_documents ===")
print(_search_documents("MCP"))

print("\n=== sql_query (cho phép) ===")
print(_sql_query("SELECT * FROM agent_metrics"))

print("\n=== sql_query (bị chặn) ===")
try:
    _sql_query("DROP TABLE agent_metrics")
except ValueError as exc:
    print(f"Đã chặn: {exc}")

print("\n=== summarize_text ===")
print("\n".join(_summarize_text("MCP build một lần. A2A kết nối agent. Routing chọn specialist.")))

=== search_documents ===
[{'id': 'doc-1', 'title': 'VinUni AI Curriculum Overview', 'body': 'Phase 2 Track 2 covers MCP, A2A, and multi-agent orchestration.'}, {'id': 'doc-2', 'title': 'MCP Transport Options', 'body': 'MCP supports stdio for local dev and HTTP+SSE for remote deployment.'}]

=== sql_query (cho phép) ===
[{'agent': 'search_agent', 'tasks_completed': 42, 'avg_latency_ms': 820}, {'agent': 'database_agent', 'tasks_completed': 31, 'avg_latency_ms': 1100}, {'agent': 'synthesis_agent', 'tasks_completed': 28, 'avg_latency_ms': 2400}]

=== sql_query (bị chặn) ===

=== summarize_text ===
- MCP build một lần
- A2A kết nối agent
- Routing chọn specialist


### Kết nối MCP Tools với ADK Agent

ADK bọc MCP server qua `McpToolset`. Agent orchestrator trong repo đã bind cả 3 tool.

In [6]:
from google.adk.tools.mcp_tool import StdioConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from mcp import StdioServerParameters

mcp_server_path = PROJECT_ROOT / "mcp_server" / "research_tools_server.py"

mcp_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="python",
            args=[str(mcp_server_path)],
        ),
        timeout=10,
    ),
    tool_filter=["search_documents", "sql_query", "summarize_text"],
)

print("✓ Đã cấu hình McpToolset")
print(f"  Server: {mcp_server_path.name}")
print(f"  Bộ lọc: search_documents, sql_query, summarize_text")

✓ Đã cấu hình McpToolset
  Server: research_tools_server.py
  Bộ lọc: search_documents, sql_query, summarize_text


C:\Users\Public\miniconda\envs\pii-env\Lib\site-packages\google\adk\features\_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [7]:
from lab_utils.governance import get_guard

guard = get_guard()

print("=== Governance SQL allowed ===")
good = guard.authorize_mcp_tool(
    "orchestrator",
    "sql_query",
    {"sql": "SELECT * FROM agent_metrics"}
)
print(good.verdict.value, "-", good.reason)

print("\n=== Governance SQL blocked ===")
bad = guard.authorize_mcp_tool(
    "orchestrator",
    "sql_query",
    {"sql": "DROP TABLE agent_metrics"}
)
print(bad.verdict.value, "-", bad.reason)

=== Governance SQL allowed ===
allow - MCP tool call được phép theo policy

=== Governance SQL blocked ===
deny - Chỉ cho phép SELECT (read-only)


### 📝 Bài tập 1.2 — Thêm tool MCP thứ tư

**Nhiệm vụ:** Mở rộng `research_tools_server.py` với tool `count_words` trả về số từ trong chuỗi.

Checklist:
- [ ] Thêm vào `list_tools()` với `inputSchema` đúng
- [ ] Triển khai trong `call_tool()`
- [ ] Thêm vào `tool_filter` trong orchestrator agent
- [ ] Test qua ADK agent hoặc gọi hàm trực tiếp

---
## Module 2 — Giao thức Agent-to-Agent (A2A)

### Lý thuyết

A2A coi agent như **microservices**:

| Khái niệm | Mô tả |
|-----------|-------|
| **Agent Card** | JSON tại `/.well-known/agent-card.json` — capability & schema |
| **Task** | Đơn vị công việc với các trạng thái vòng đời |
| **Message** | Giao tiếp trong một task |

### Vòng đời Task

```
Submitted → Working → Input Required → Completed
                  ↘ Failed / Canceled
```

Task có thể **tạm dừng** ở `input-required` — agent yêu cầu caller cung cấp thêm thông tin trước khi tiếp tục.

### Ánh xạ sang ADK

| Hành động | API ADK |
|----------|--------|
| Expose agent làm A2A server | `to_a2a(agent, port=8001)` |
| Tiêu thụ agent remote | `RemoteA2aAgent(name, description, agent_card=url)` |
| Thay thế qua CLI | `adk api_server --a2a --port 8001 agents/search_agent` |

### 🚀 Thực hành — Kiểm tra lại A2A (Module 2)

> **Đã khởi động ở Module 0.5 chưa?** Nếu chưa, quay lại phầ **「Bước bắt buộc — Khởi động A2A Specialist Servers」** và chạy cell khởi động + kiểm tra.

Cell bên dưới lặp lại bước xác nhận agent card (giống Module 0.5).

In [8]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "mcp_server"))

from research_tools_server import _count_words
from lab_utils.governance import get_guard

print("=== count_words ===")
print(_count_words("MCP A2A routing chọn specialist agent"))

guard = get_guard()
print("\n=== Allowed MCP tools ===")
print(guard.get_allowed_mcp_tools("orchestrator"))

decision = guard.authorize_mcp_tool(
    "orchestrator",
    "count_words",
    {"text": "MCP A2A routing chọn specialist agent"},
)
print("\n=== Governance count_words ===")
print(decision.verdict.value, "-", decision.reason)

=== count_words ===
6

=== Allowed MCP tools ===
['search_documents', 'sql_query', 'summarize_text', 'count_words']

=== Governance count_words ===
allow - MCP tool call được phép theo policy


In [9]:
import httpx

AGENT_CARDS = {
    "search": "http://localhost:8001/.well-known/agent-card.json",
    "database": "http://localhost:8002/.well-known/agent-card.json",
}

for label, url in AGENT_CARDS.items():
    try:
        response = httpx.get(url, timeout=3)
        response.raise_for_status()
        card = response.json()
        print(f"✓ {label}_agent: {card.get('name')} — {card.get('description', '')[:60]}...")
    except Exception as exc:
        print(f"✗ Không kết nối được {label}_agent: {exc}")
        print(f"  Khởi động: uvicorn agents.{label}_agent.agent:a2a_app --port {8001 if label == 'search' else 8002}")

✓ search_agent: search_agent — Tìm kiếm web và trả về đoạn trích liên quan cho tác vụ nghiê...
✓ database_agent: database_agent — Truy vấn database chỉ đọc và trả về metrics có cấu trúc....


### Tiêu thụ Remote Agent từ Orchestrator

Orchestrator dùng `RemoteA2aAgent` làm sub-agent. LLM coordinator của ADK ủy quyền dựa trên `description` của từng sub-agent.

In [10]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

remote_search = RemoteA2aAgent(
    name="search_agent",
    description="Tìm kiếm tài liệu và web để thu thập bằng chứng nghiên cứu.",
    agent_card=AGENT_CARDS["search"],
)

remote_database = RemoteA2aAgent(
    name="database_agent",
    description="Chạy SQL chỉ đọc trên bảng metrics của agent.",
    agent_card=AGENT_CARDS["database"],
)

print("✓ Đã cấu hình proxy RemoteA2aAgent")
print(f"  search → {AGENT_CARDS['search']}")
print(f"  database → {AGENT_CARDS['database']}")

✓ Đã cấu hình proxy RemoteA2aAgent
  search → http://localhost:8001/.well-known/agent-card.json
  database → http://localhost:8002/.well-known/agent-card.json


C:\Users\THAI NHI\AppData\Local\Temp\ipykernel_16908\3397977623.py:3: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subject to breaking changes. A2A protocol and SDK are themselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_search = RemoteA2aAgent(
C:\Users\THAI NHI\AppData\Local\Temp\ipykernel_16908\3397977623.py:9: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subject to breaking changes. A2A protocol and SDK are themselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_database = RemoteA2aAgent(


### 📝 Bài tập 2.1 — A2A vs Sub-Agent Local

| Tiêu chí | A2A (Remote) | Sub-Agent Local |
|----------|-------------|------------------|
| Triển khai | Mỗi agent chạy như một service riêng qua HTTP/A2A, có `agent-card.json`, port riêng và có thể deploy độc lập. | Agent chạy cùng process với orchestrator trong cùng codebase, thường khai báo trực tiếp trong `sub_agents=[...]`. |
| Hiệu năng | Chậm hơn một chút do có network/HTTP overhead và cần serialize message/task. | Nhanh hơn vì gọi nội bộ trong cùng runtime, ít overhead hơn. |
| Cô lập state | Cô lập tốt hơn: mỗi agent có runtime, log, health check và state riêng; dễ restart/scale từng agent. | State dễ chia sẻ hơn nhưng coupling cao hơn; lỗi ở một agent có thể ảnh hưởng cả process orchestrator. |
| Phù hợp khi | Hệ multi-agent production, nhiều specialist, cần scale riêng từng agent, cần observability, audit, health check và ownership rõ ràng. | Prototype nhanh, demo nhỏ, ít agent, chạy local đơn giản, chưa cần tách service hoặc scale độc lập. |

**Thảo luận:**  
Nên chọn A2A thay vì `sub_agents=[local_agent]` khi agent cần hoạt động như một microservice độc lập: có endpoint riêng, agent card riêng, capability metadata rõ ràng, health check riêng, log/audit riêng và có thể scale/restart/deploy độc lập. A2A phù hợp hơn cho hệ multi-agent thật, nơi orchestrator chỉ điều phối còn các specialist có thể do các team khác nhau phát triển hoặc chạy ở môi trường khác nhau.

Ngược lại, nếu chỉ làm prototype nhỏ, demo nhanh hoặc số lượng agent ít, `sub_agents=[local_agent]` đơn giản hơn vì không cần mở port, không cần HTTP server và ít lỗi hạ tầng hơn.

---
## Module 3 — Mẫu Agentic Routing

### So sánh chiến lược routing

| Chiến lược | Ưu điểm | Nhược điểm | Khi nào dùng |
|------------|---------|------------|-------------|
| **Keyword** | Nhanh, đơn giản | Dễ vỡ, bỏ sót ngữ cảnh | ≤5 agent |
| **Embedding** | Bền, scale tốt | Chậm hơn, cần embedding | 5–50 agent |
| **LLM-based** | Linh hoạt, hiểu ngữ cảnh | Đắt, chậm | Routing phức tạp |

**Semantic routing:** embed request → cosine similarity với mô tả capability của agent → top-k ứng viên.

**Fallback chain:** agent chính → agent dự phòng → leo thang lên người — không để user bị kẹt.

In [11]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.semantic_router import AgentCapability, SemanticRouter

router = SemanticRouter(
    agents=[
        AgentCapability(
            name="search_agent",
            description="Tìm kiếm web tài liệu nghiên cứu bằng chứng",
            tags=["search", "web", "documents"],
        ),
        AgentCapability(
            name="database_agent",
            description="SQL metrics phân tích database truy vấn SELECT",
            tags=["sql", "metrics", "database"],
        ),
        AgentCapability(
            name="synthesis_agent",
            description="Tóm tắt kết hợp kết quả thành báo cáo cuối",
            tags=["summary", "report", "synthesis"],
        ),
    ]
)

test_queries = [
    "Tìm bài viết về việc áp dụng giao thức MCP",
    "SELECT độ trễ trung bình từ agent_metrics",
    "Viết tóm tắt một trang về kết quả nghiên cứu",
    "Xin chào, bạn làm được gì?",
]

print(f"{'Truy vấn':<50} {'Định tuyến tới':<20} Điểm")
print("-" * 80)
for query in test_queries:
    candidates = router.route(query, top_k=1)
    target = router.route_with_fallback(query, fallback="orchestrator")
    score = candidates[0][1] if candidates else 0.0
    print(f"{query[:48]:<50} {target:<20} {score:.3f}")

Truy vấn                                           Định tuyến tới       Điểm
--------------------------------------------------------------------------------
Tìm bài viết về việc áp dụng giao thức MCP         synthesis_agent      0.534
SELECT độ trễ trung bình từ agent_metrics          synthesis_agent      0.365
Viết tóm tắt một trang về kết quả nghiên cứu       synthesis_agent      0.758
Xin chào, bạn làm được gì?                         search_agent         0.365


### Mẫu Agent Registry

Trên production, agent đăng ký khi khởi động và hủy đăng ký khi tắt. Registry lưu metadata capability để discovery.

In [12]:
from lab_utils.agent_registry import AgentRegistry, RegisteredAgent

registry = AgentRegistry()

registry.register(
    RegisteredAgent(
        name="search_agent",
        url="http://localhost:8001",
        description="Chuyên gia tìm kiếm web và tài liệu",
        capabilities={"skills": ["search_web"], "transport": "a2a"},
    )
)
registry.register(
    RegisteredAgent(
        name="database_agent",
        url="http://localhost:8002",
        description="Chuyên gia SQL metrics chỉ đọc",
        capabilities={"skills": ["run_sql_query"], "transport": "a2a"},
    )
)

print("Các agent đã đăng ký:")
for agent in registry.list_agents():
    status = "khỏe mạnh" if agent.healthy else "không khỏe"
    print(f"  • {agent.name} ({status}) @ {agent.url}")

print("\nTìm capability 'sql':")
for agent in registry.find_by_capability("sql"):
    print(f"  → {agent.name}")

Các agent đã đăng ký:
  • search_agent (khỏe mạnh) @ http://localhost:8001
  • database_agent (khỏe mạnh) @ http://localhost:8002

Tìm capability 'sql':
  → database_agent


### 📝 Bài tập 3.1 — Xây dựng Fallback Chain

**Nhiệm vụ:** Mở rộng `SemanticRouter.route_with_fallback` để nhận danh sách fallback có thứ tự:

```python
def route_with_chain(self, request: str, chain: list[str]) -> str:
    """Thử route chính; nếu điểm < ngưỡng, đi theo chuỗi fallback."""
    ...
```

Test với: `chain=["search_agent", "database_agent", "orchestrator"]`

In [1]:
import importlib
import lab_utils.semantic_router as router_module

router_module = importlib.reload(router_module)

AgentCapability = router_module.AgentCapability
SemanticRouter = router_module.SemanticRouter

router = SemanticRouter(
    agents=[
        AgentCapability(
            name="search_agent",
            description="Tìm kiếm web tài liệu nghiên cứu bằng chứng MCP bài viết",
            tags=["search", "web", "documents", "mcp", "tìm kiếm"],
        ),
        AgentCapability(
            name="database_agent",
            description="SQL metrics phân tích database truy vấn SELECT agent_metrics độ trễ latency",
            tags=["sql", "metrics", "database", "select", "agent_metrics"],
        ),
        AgentCapability(
            name="synthesis_agent",
            description="Tóm tắt kết hợp kết quả thành báo cáo cuối executive summary",
            tags=["summary", "report", "synthesis", "tóm tắt", "báo cáo"],
        ),
    ]
)

chain = ["search_agent", "database_agent", "orchestrator"]

for query in [
    "SELECT độ trễ trung bình từ agent_metrics",
    "Tìm bài viết về MCP",
    "Viết tóm tắt báo cáo kết quả nghiên cứu",
    "Câu rất mơ hồ không có keyword rõ",
]:
    print(query, "->", router.route_with_chain(query, chain))

SELECT độ trễ trung bình từ agent_metrics -> database_agent
Tìm bài viết về MCP -> search_agent
Viết tóm tắt báo cáo kết quả nghiên cứu -> synthesis_agent
Câu rất mơ hồ không có keyword rõ -> search_agent


---
## Module 4 — Demo Full Luồng (chạy được)

### Mẫu Orchestrator

```
Yêu cầu người dùng
     │
     ▼
Orchestrator (phân rã + định tuyến)
     ├──► Search Agent (A2A :8001)
     ├──► Database Agent (A2A :8002)
     └──► MCP Tools (stdio: search / sql / summarize)
```

**Điều kiện:** Module 0.5 đã chạy — A2A servers ✓ trên cổng 8001/8002.

Hai ví dụ bên dưới chạy **orchestrator → A2A + MCP → tổng hợp** qua `run_full_flow()`.

In [13]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.full_flow import check_a2a_servers, print_flow_summary, run_full_flow

ok, errors = check_a2a_servers()
if not ok:
    raise RuntimeError(
        "A2A servers chưa sẵn sàng — quay lại Module 0.5.\n" + "\n".join(errors)
    )
print("✓ A2A servers OK — sẵn sàng chạy full luồng")

✓ A2A servers OK — sẵn sàng chạy full luồng


### Ví dụ 1 — A2A: Orchestrator → Search Agent

Luồng: user → orchestrator → **transfer A2A** → `search_agent` → tổng hợp.

In [14]:
# Ví dụ 1 — A2A search delegation
result_1 = await run_full_flow(
    "Transfer sang search_agent để tìm kiếm web về multi-agent orchestration.",
    verbose=True,
)
print_flow_summary(result_1)

Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


[orchestrator] Tôi sẽ chuyển yêu cầu của bạn sang search_agent để thực hiện tìm kiếm thông tin về multi-agent orchestration.


[search_agent] Dưới đây là thông tin tổng quan về **Multi-Agent Orchestration** (Điều phối đa tác nhân) dựa trên tìm kiếm web:

### 1. Multi-Agent Orchestration là gì?
Multi-agent orchestration là quá trình quản lý, điều phối và thiết lập sự tương tác giữa nhiều tác nhân AI (AI agents) để cùng nhau giải quyết các ...

Trace ID : adbd15b5-8297-4989-a035-aa7bba548635
Agents   : orchestrator, search_agent
Sự kiện  : 2 lượt
Dưới đây là thông tin tổng quan về **Multi-Agent Orchestration** (Điều phối đa tác nhân) dựa trên tìm kiếm web:

### 1. Multi-Agent Orchestration là gì?
Multi-agent orchestration là quá trình quản lý, điều phối và thiết lập sự tương tác giữa nhiều tác nhân AI (AI agents) để cùng nhau giải quyết các nhiệm vụ phức tạp mà một tác nhân đơn lẻ khó có thể thực hiện hiệu quả. Thay vì một mô hình ngôn ngữ lớn (LLM) làm tất cả, hệ thống này chia nhỏ công

### Ví dụ 2 — MCP + A2A: Tài liệu + Metrics + Tổng hợp

Luồng: user → orchestrator → **MCP** `search_documents` + `sql_query` → tổng hợp báo cáo.

*(Orchestrator cũng có thể ủy quyền `database_agent` qua A2A cho bước SQL — thử đổi prompt nếu muốn.)*

In [15]:
# Ví dụ 2 — MCP multi-tool + tổng hợp
result_2 = await run_full_flow(
    "Bước 1: dùng search_documents tìm MCP. "
    "Bước 2: dùng sql_query SELECT * FROM agent_metrics. "
    "Bước 3: tóm tắt kết quả thành báo cáo ngắn.",
    verbose=True,
)
print_flow_summary(result_2)

Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


[orchestrator] Tôi sẽ hỗ trợ bạn thực hiện chuỗi tác vụ này theo từng bước, bắt đầu bằng việc tìm kiếm tài liệu về MCP.


[search_agent] Xin lỗi, tôi không có quyền truy cập vào cơ sở dữ liệu SQL (như `agent_metrics`) để thực hiện bước 2. Tôi là một chuyên gia tìm kiếm web, vì vậy tôi chỉ có thể hỗ trợ bạn tìm kiếm thông tin công khai trên internet.

Dưới đây là báo cáo tóm tắt cho Bước 1:

### Báo cáo: Model Context Protocol (MCP)

...

Trace ID : 501effdc-f2f5-461a-9014-5216a6d4f7bd
Agents   : orchestrator, search_agent
Sự kiện  : 2 lượt
Xin lỗi, tôi không có quyền truy cập vào cơ sở dữ liệu SQL (như `agent_metrics`) để thực hiện bước 2. Tôi là một chuyên gia tìm kiếm web, vì vậy tôi chỉ có thể hỗ trợ bạn tìm kiếm thông tin công khai trên internet.

Dưới đây là báo cáo tóm tắt cho Bước 1:

### Báo cáo: Model Context Protocol (MCP)

**Nguồn:** Kết quả tìm kiếm web

**Tóm tắt:**
*   **Định nghĩa:** MCP (Model Context Protocol) là một giao thức chuẩn hóa được thiết kế để kết nối các hệ 

### Capstone Demo — ADK Web UI (sinh viên tự chạy)

ADK Web là giao diện chat tương tác cho orchestrator. Khác với `run_full_flow()` trong notebook, bạn gõ prompt trực tiếp tại trình duyệt.

#### Kiến trúc khi chạy capstone

```
localhost:8000  ADK Web (orchestrator)
     ├── A2A → search_agent      :8001
     ├── A2A → database_agent    :8002
     ├── A2A → synthesis_agent   :8003
     └── MCP   research_tools    (stdio, spawn tự động)
```

#### Cách khởi động (khuyến nghị — một lệnh)

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD
bash scripts/start_capstone.sh
```

Script trên sẽ: (1) khởi động 3 A2A specialist nền, (2) mở ADK Web tại http://localhost:8000.

#### Cách khởi động thủ công (2 terminal)

**Terminal 1 — A2A specialists:**
```bash
bash scripts/start_a2a_servers.sh
```

**Terminal 2 — ADK Web:**
```bash
bash scripts/start_adk_web.sh
# hoặc: adk web agents/orchestrator
```

```bash
# ❌ SAI — sẽ báo "No root_agent found for 'agents'"
adk web agents
```

#### Công cụ mới trong capstone

| Thành phần | File | Vai trò |
|------------|------|--------|
| `start_capstone.sh` | `scripts/` | Khởi động A2A + ADK Web một lệnh |
| `synthesis_agent` | `agents/synthesis_agent/` | Specialist tổng hợp báo cáo (:8003) |
| `suggest_routing` | `lab_utils/routing_tool.py` | Gợi ý semantic routing (tư vấn) |
| Auto `trace_id` | `adk_callbacks.py` | Tự sinh trace cho ADK Web (governance A2A) |

Chạy cell **kiểm tra trước khi mở ADK Web** bên dưới, rồi làm bài tập ghi kết quả.

In [16]:
# Kiểm tra trước khi mở ADK Web — chạy cell này trong notebook
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import httpx
from lab_utils.routing_tool import suggest_routing

CAPSTONE_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

print("=== Kiểm tra A2A specialists ===")
all_ok = True
for name, url in CAPSTONE_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        print(f"✓ {name}")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")

print("\n=== Demo suggest_routing (orchestrator tool) ===")
for q in [
    "Tìm bài viết về MCP",
    "SELECT avg latency FROM agent_metrics",
    "Viết báo cáo tổng hợp kết quả nghiên cứu",
]:
    result = suggest_routing(q)
    print(f"  '{q[:40]}...' → {result['recommended_agent']} (top score: {result['top_candidates'][0]['score']})")

if all_ok:
    print("\n✓ Sẵn sàng — mở terminal và chạy: bash scripts/start_capstone.sh")
    print("  Sau đó truy cập http://localhost:8000")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

=== Kiểm tra A2A specialists ===
✓ search_agent
✓ database_agent
✓ synthesis_agent

=== Demo suggest_routing (orchestrator tool) ===
  'Tìm bài viết về MCP...' → synthesis_agent (top score: 0.606)
  'SELECT avg latency FROM agent_metrics...' → database_agent (top score: 0.261)
  'Viết báo cáo tổng hợp kết quả nghiên cứu...' → synthesis_agent (top score: 0.766)

✓ Sẵn sàng — mở terminal và chạy: bash scripts/start_capstone.sh
  Sau đó truy cập http://localhost:8000


### 📝 Bài tập sinh viên — Ghi kết quả ADK Web

**Hướng dẫn:** Sau khi `start_capstone.sh` chạy, mở http://localhost:8000. **Tạo session mới** (nút +), gõ từng prompt bên dưới.

**Đọc kết quả trong ADK Web:**
| Bubble | Ý nghĩa |
|--------|---------|
| `State: trace_id, task_id...` | Governance khởi tạo session — **không phải câu trả lời** |
| Text từ orchestrator | Câu trả lời chính |
| Tab **Trace** (bên phải) | Xem `transfer_to_agent`, MCP calls, A2A events |

Nếu chat trống sau prompt → mở **Trace**, kiểm tra có `transfer_to_agent` không. Restart: `Ctrl+C` rồi `bash scripts/start_capstone.sh`.

| # | Prompt (copy vào ADK Web) | Kỳ vọng |
|---|---------------------------|---------|
| **W1** | `Tôi cần tìm web về multi-agent orchestration. Hãy transfer_to_agent sang search_agent và trả kết quả.` | A2A → search_agent + text |
| **W2** | `Bước 1: dùng search_documents tìm MCP. Bước 2: dùng sql_query SELECT * FROM agent_metrics. Bước 3: tóm tắt báo cáo ngắn.` | MCP: search_documents + sql_query |
| **W3** | `Ủy quyền synthesis_agent tổng hợp báo cáo executive từ các findings về MCP và A2A.` | A2A → synthesis_agent |
| **W4** | `Gọi suggest_routing rồi giải thích bạn sẽ chọn agent nào: "SELECT độ trễ trung bình từ agent_metrics"` | Tool suggest_routing → database_agent |
| **W5** | `DROP TABLE agent_metrics` (thử governance) | Bị chặn — blocked / deny |

**Quan sát thêm (ghi vào bảng):**
- Trace ID có trong session state không?
- Agent nào xuất hiện trong Trace? (`orchestrator`, `search_agent`, …)
- `tail -5 logs/governance_audit.jsonl`

In [18]:
# 📝 SINH VIÊN ĐIỀN KẾT QUẢ ADK WEB — thay thế placeholder bằng quan sát thực tế

adk_web_results = [
    {
        "prompt_id": "W1",
        "agents_involved": ["orchestrator", "search_agent"],
        "tools_or_protocol": "A2A → search_agent",
        "outcome": "ĐẠT",
        "notes": "Trace hiển thị transfer_to_agent và orchestrator -> search_agent; search_agent trả kết quả về multi-agent orchestration.",
    },
    {
        "prompt_id": "W2",
        "agents_involved": ["orchestrator", "synthesis_agent"],
        "tools_or_protocol": "MCP (search_documents, sql_query) + A2A → synthesis_agent",
        "outcome": "ĐẠT",
        "notes": "Trace hiển thị search_documents, sql_query và transfer_to_agent sang synthesis_agent; kết quả có executive summary và metrics agent.",
    },
    {
        "prompt_id": "W3",
        "agents_involved": ["orchestrator", "synthesis_agent"],
        "tools_or_protocol": "A2A → synthesis_agent",
        "outcome": "ĐẠT",
        "notes": "Trace hiển thị transfer_to_agent và orchestrator -> synthesis_agent; synthesis_agent trả executive summary và key points.",
    },
    {
        "prompt_id": "W4",
        "agents_involved": ["orchestrator", "database_agent"],
        "tools_or_protocol": "suggest_routing + A2A → database_agent",
        "outcome": "ĐẠT",
        "notes": "Trace hiển thị suggest_routing; orchestrator chọn database_agent vì yêu cầu là SQL/metrics; database_agent trả latency trung bình 1440ms.",
    },
    {
        "prompt_id": "W5",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "MCP sql_query — governance deny",
        "outcome": "ĐẠT",
        "notes": "Trace hiển thị sql_query với DROP TABLE agent_metrics; governance trả blocked/deny vì chỉ cho phép SELECT read-only.",
    },
]

print(f"{'ID':<4} {'Agents':<35} {'Protocol':<45} {'Kết quả':<12} Ghi chú")
print("-" * 135)
for row in adk_web_results:
    agents = ", ".join(row["agents_involved"])
    print(f"{row['prompt_id']:<4} {agents:<35} {row['tools_or_protocol']:<45} {row['outcome']:<12} {row['notes']}")

passed = sum(1 for r in adk_web_results if r["outcome"] == "ĐẠT")
print(f"\nTổng: {passed}/{len(adk_web_results)} prompt đạt yêu cầu")
print("\n💡 Đã chụp screenshot ADK Web Events + Traces cho W1–W5, đặc biệt tab Traces cạnh tab Events theo yêu cầu.")

ID   Agents                              Protocol                                      Kết quả      Ghi chú
---------------------------------------------------------------------------------------------------------------------------------------
W1   orchestrator, search_agent          A2A → search_agent                            ĐẠT          Trace hiển thị transfer_to_agent và orchestrator -> search_agent; search_agent trả kết quả về multi-agent orchestration.
W2   orchestrator, synthesis_agent       MCP (search_documents, sql_query) + A2A → synthesis_agent ĐẠT          Trace hiển thị search_documents, sql_query và transfer_to_agent sang synthesis_agent; kết quả có executive summary và metrics agent.
W3   orchestrator, synthesis_agent       A2A → synthesis_agent                         ĐẠT          Trace hiển thị transfer_to_agent và orchestrator -> synthesis_agent; synthesis_agent trả executive summary và key points.
W4   orchestrator, database_agent        suggest_routing + A2A → dat

## Module 5 — State, Bảo mật & Observability

### Quản lý State

| Mẫu | Trường hợp dùng |
|-----|----------------|
| **Agent stateless** | Scale ngang, K8s auto-scale |
| **State ngoài (Redis)** | Ngữ cảnh ngắn hạn (<1 giờ) |
| **PostgreSQL** | Lịch sử hội thoại lâu dài |
| **Session ID** | Sticky routing khi stateful |

**Thực hành tốt:** Thiết kế stateless trước; externalize state khi cần.

Trong ADK, dùng `tool_context.state` trong function tool để chia sẻ dữ liệu giữa các lượt.

In [30]:
# Ví dụ: state qua ToolContext (mẫu khái niệm)
example_tool = '''
from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"
'''
print(example_tool)


from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"



### Bảo mật & Governance

| Kiểm soát | Triển khai |
|-----------|------------|
| **Rate limiting** | Giới hạn gọi tool theo agent & user |
| **Sandbox execution** | Code agent trong Docker, không network |
| **Human-in-the-loop** | Hành động quan trọng cần phê duyệt |
| **Audit log** | Timestamp, agent ID, I/O cho mọi lần gọi tool |
| **Minimal capability** | Agent chỉ có tool cần thiết |

**Chống chạy vô hạn:** tối đa 50 lần gọi tool/task, tối đa 5 phút thực thi, trần chi phí mỗi request.

### Ma trận Capability Agent

| Agent | Được phép | Bị chặn |
|-------|-----------|--------|
| Research/Search | Search, đọc tài liệu/web | Ghi, xóa, gửi email |
| Database | Chỉ SELECT/read-only | DDL, DML, DROP, DELETE, UPDATE |
| Synthesis | Tổng hợp findings thành báo cáo | Thu thập dữ liệu mới, ghi/xóa/gửi email |
| Orchestrator | Điều phối, gọi MCP/A2A trong whitelist | Gọi agent/tool ngoài policy |

### Data Governance — MCP & A2A

Hệ thống governance nằm trong `lab_utils/governance/`:

| Thành phần | Vai trò |
|------------|--------|
| `policy.json` | Ma trận capability: ai được gọi MCP/A2A tool nào |
| `GovernanceGuard` | Kiểm tra trước mỗi lần gọi: SQL read-only, rate limit, PII, blocked keyword |
| `AuditLogger` | Ghi audit vào `logs/governance_audit.jsonl` |
| `adk_callbacks.py` | `before_tool_callback` chặn tool vi phạm policy |

**Luồng kiểm soát:**

```text
Request → GovernanceGuard → [ALLOW | DENY | HITL_REQUIRED]
                ↓
         AuditLogger(timestamp, actor, I/O)
                ↓
         MCP tool / A2A dispatch
```


### Data Governance — MCP & A2A (thực hành)

Hệ thống governance nằm trong `lab_utils/governance/`:

| Thành phần | Vai trò |
|------------|--------|
| `policy.json` | Ma trận capability: ai được gọi MCP/A2A tool nào |
| `GovernanceGuard` | Kiểm tra trước mỗi lần gọi (SQL read-only, rate limit, PII) |
| `AuditLogger` | Ghi audit vào `logs/governance_audit.jsonl` |
| `adk_callbacks.py` | `before_tool_callback` chặn tool vi phạm policy |

**Luồng kiểm soát:**

```
Request → GovernanceGuard → [ALLOW | DENY | HITL_REQUIRED]
                ↓
         AuditLogger (timestamp, actor, I/O)
                ↓
         MCP tool / A2A dispatch
```

### 📝 Bài tập 5.1 — Thiết kế chính sách Governance

| Agent | Tool / Capability được phép | Hành động bị chặn | Cần HITL khi nào | Rate limit đề xuất | Giới hạn task |
|---|---|---|---|---|---|
| `orchestrator` | Điều phối request, gọi MCP tools (`search_documents`, `sql_query`, `summarize_text`, `count_words`), dispatch A2A tới `search_agent`, `database_agent`, `synthesis_agent` | Không được gọi agent ngoài whitelist; không tự thực hiện SQL ghi/DDL/DML; không gửi email/deploy production | Dispatch A2A không có `trace_id`; request vượt trần chi phí; request chứa PII hoặc hành động rủi ro cao | 30 calls/phút | Tối đa 50 tool calls/task, tối đa 300 giây/task |
| `search_agent` | `search_web`, đọc/tìm kiếm tài liệu hoặc web | Ghi/xóa dữ liệu, gửi email, sửa database, chạy SQL | Search query có PII hoặc keyword nhạy cảm như `password`, credential, secret | 30 calls/phút | Chỉ đọc, không ghi; trả kết quả kèm nguồn/snippet |
| `database_agent` | `run_sql_query` với truy vấn `SELECT` trên bảng được phép như `agent_metrics` | `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `CREATE`, `TRUNCATE` và bảng ngoài whitelist | Query chứa email, SSN, PII hoặc truy vấn dữ liệu confidential nhạy cảm | 30 calls/phút | Chỉ SELECT, read-only, audit mọi query |
| `synthesis_agent` | `synthesize_report`, tổng hợp findings thành executive summary và key points | Không thu thập dữ liệu mới, không gọi SQL/search trực tiếp, không ghi/xóa/gửi email | Báo cáo chứa PII, thông tin nhạy cảm hoặc yêu cầu publish/gửi ra ngoài | 30 calls/phút | Chỉ tổng hợp input đã có, không tạo hành động bên ngoài |

**Giải thích:**  
Hệ multi-agent cần governance vì mỗi agent có capability khác nhau. `orchestrator` chỉ nên điều phối và gọi tool/agent trong whitelist. `database_agent` phải bị giới hạn read-only để tránh làm hỏng dữ liệu. `search_agent` chỉ đọc/tìm kiếm, không ghi. `synthesis_agent` chỉ tổng hợp dữ liệu đã có. Tất cả MCP/A2A actions cần audit log với `trace_id` để quan sát toàn bộ luồng request.

**Policy thực tế trong repo:**  
Repo đang đặt giới hạn toàn cục là `max_tool_calls_per_task = 50`, `max_execution_seconds = 300`, `cost_ceiling_usd = 10.0`, `rate_limit_per_minute = 30`. MCP `research-tools` dùng `stdio`, chỉ cho `orchestrator` gọi, và A2A chỉ cho orchestrator dispatch tới các specialist trong whitelist.

In [31]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.governance import get_guard, AuditLogger

guard = get_guard()
audit = AuditLogger()

print("=== Ma trận MCP (orchestrator) ===")
conn = guard.authorize_mcp_connection("orchestrator")
print(f"Kết nối MCP: {conn.verdict.value} — tools: {conn.metadata.get('allowed_tools')}")

print("\n=== Kiểm tra vi phạm SQL (MCP) ===")
bad_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "DROP TABLE agent_metrics"}
)
print(f"DROP TABLE → {bad_sql.verdict.value}: {bad_sql.reason}")

good_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "SELECT * FROM agent_metrics"}
)
print(f"SELECT hợp lệ → {good_sql.verdict.value}")

print("\n=== Kiểm tra A2A dispatch ===")
allowed = guard.authorize_a2a_dispatch(
    "orchestrator", "search_agent", trace_id="demo-trace-001"
)
print(f"orchestrator → search_agent: {allowed.verdict.value}")

blocked = guard.authorize_a2a_dispatch(
    "orchestrator", "email_agent", trace_id="demo-trace-001"
)
print(f"orchestrator → email_agent: {blocked.verdict.value}: {blocked.reason}")

no_trace = guard.authorize_a2a_dispatch("orchestrator", "database_agent")
print(f"Không có trace_id → {no_trace.verdict.value}: {no_trace.reason}")

print("\n=== PII → HITL ===")
pii_sql = guard.authorize_mcp_tool(
    "orchestrator",
    "sql_query",
    {"sql": "SELECT * FROM agent_metrics WHERE email = 'user@vinuni.edu.vn'"},
)
print(f"PII trong SQL → {pii_sql.verdict.value}: {pii_sql.reason}")

print("\n=== Tóm tắt audit log ===")
print(audit.summary())

=== Ma trận MCP (orchestrator) ===
Kết nối MCP: allow — tools: ['search_documents', 'sql_query', 'summarize_text', 'count_words']

=== Kiểm tra vi phạm SQL (MCP) ===
DROP TABLE → deny: Chỉ cho phép SELECT (read-only)
SELECT hợp lệ → allow

=== Kiểm tra A2A dispatch ===
orchestrator → search_agent: allow
orchestrator → email_agent: deny: 'orchestrator' không được dispatch A2A tới 'email_agent'. Chỉ cho phép: ['search_agent', 'database_agent', 'synthesis_agent']
Không có trace_id → hitl_required: A2A dispatch yêu cầu trace_id trong metadata (W3C Trace Context)

=== PII → HITL ===
PII trong SQL → hitl_required: Phát hiện dữ liệu nhạy cảm (PII) — cần phê duyệt HITL

=== Tóm tắt audit log ===
{'allow': 45, 'deny': 8, 'hitl_required': 4}


### 📝 Bài tập 5.2 — Mở rộng chính sách governance

**Yêu cầu:**
1. Thêm `synthesis_agent` vào `allowed_targets` của `orchestrator`.
2. Thêm rule chặn từ khóa `password` trong `search_documents`.
3. Chạy lại governance test và xác nhận audit log có đủ `deny` / `hitl_required`.

**Kết quả thực hiện:**  
- `synthesis_agent` đã có trong `allowed_targets` của `orchestrator`.
- Đã thêm `blocked_keywords: ["password"]` vào policy của MCP tool `search_documents`.
- Đã cập nhật `GovernanceGuard.authorize_mcp_tool()` để deny query chứa keyword bị chặn.
- Test với query `"find password leak in logs"` trả về `deny - Truy vấn chứa từ khóa bị chặn: password`.

In [32]:
# Reload module guard để Jupyter nhận code mới sau khi sửa guard.py
import importlib
import lab_utils.governance.guard as guard_module

guard_module = importlib.reload(guard_module)

GovernanceGuard = guard_module.GovernanceGuard
AuditLogger = guard_module.AuditLogger

guard = GovernanceGuard()
audit = AuditLogger()

print("=== Test blocked keyword password ===")
blocked_password = guard.authorize_mcp_tool(
    "orchestrator",
    "search_documents",
    {"query": "find password leak in logs"},
    trace_id="demo-password-block",
)
print(blocked_password.verdict.value, "-", blocked_password.reason)

print("\n=== Audit summary ===")
print(audit.summary())

=== Test blocked keyword password ===
deny - Truy vấn chứa từ khóa bị chặn: password

=== Audit summary ===
{'allow': 45, 'deny': 9, 'hitl_required': 4}


### Observability — Distributed Tracing

Một **trace ID** duy nhất nên bao trùm toàn bộ request multi-agent:

```text
Trace ID: abc-123
└── Orchestrator
    ├── Search Agent
    │   └── search_web / A2A events
    ├── Database Agent
    │   └── run_sql_query
    └── MCP Tools
        ├── search_documents
        └── sql_query
```

ADK truyền metadata qua ranh giới A2A qua `RunConfig.custom_metadata`. Trong ADK Web, tab **Traces** đã được dùng để quan sát `transfer_to_agent`, MCP calls và A2A events cho W1–W5.

In [33]:
import uuid
from google.adk.agents.run_config import RunConfig

trace_id = str(uuid.uuid4())
run_config = RunConfig(
    custom_metadata={
        "trace_id": trace_id,
        "lab": "day26",
        "user_tier": "student",
    }
)

print(f"Trace ID: {trace_id}")
print(f"Các khóa metadata: {list(run_config.custom_metadata.keys())}")
print("\nTrên A2A agent remote, metadata xuất hiện dạng:")
print('  event.custom_metadata["a2a_metadata"]["trace_id"]')

Trace ID: 75aceac4-8e66-488d-9ff8-570494d1b287
Các khóa metadata: ['trace_id', 'lab', 'user_tier']

Trên A2A agent remote, metadata xuất hiện dạng:
  event.custom_metadata["a2a_metadata"]["trace_id"]


### Các metric cần theo dõi

| Metric | Mục đích |
|--------|----------|
| `tasks_completed` / `tasks_failed` | Độ tin cậy |
| `avg_task_duration` | Độ trễ |
| `tool_call_count` | Phát hiện chạy vô hạn |
| `cost_per_task` | Phân bổ ngân sách |
| `queue_depth` | Lập kế hoạch năng lực |

---

## Lab #26 Capstone — Checklist kết quả

- [x] MCP server với 3 tool gốc qua stdio: `search_documents`, `sql_query`, `summarize_text`
- [x] Mở rộng MCP tool thứ tư: `count_words`
- [x] Agent registry có health check/capability discovery
- [x] Semantic router + `suggest_routing` tool trên orchestrator
- [x] Search agent expose qua A2A cổng 8001
- [x] Database agent expose qua A2A cổng 8002
- [x] Synthesis agent expose qua A2A cổng 8003
- [x] Orchestrator consume specialist qua `RemoteA2aAgent`
- [x] ADK Web demo W1–W5 đạt 5/5
- [x] Screenshot ADK Web đã chụp Events + Traces, đặc biệt tab Traces cạnh Events
- [x] Trace ID/session trace xuất hiện trong ADK Web và audit log
- [x] Governance policy ghi audit vào `logs/governance_audit.jsonl`
- [x] Governance deny: `DROP TABLE agent_metrics` bị chặn
- [x] HITL case: A2A dispatch không có `trace_id` và SQL có PII trả `hitl_required`
- [x] Mở rộng policy: chặn keyword `password` trong `search_documents`

---

## Tổng kết — Điểm then chốt

1. **MCP** chuẩn hóa giao diện tool — build một lần, dùng trên nhiều LLM framework.
2. **A2A** là microservices cho AI agent — có agent card, task, message, trace và capability contract.
3. **Agentic routing** giúp orchestrator chọn đúng specialist thay vì hard-code agent.
4. **Governance + Observability** là phần bắt buộc khi multi-agent có quyền gọi tool, gọi SQL hoặc dispatch sang agent khác.

## Xem trước buổi sau

**Ngày 27:** Data Observability & Advanced Monitoring — Great Expectations, phát hiện bất thường, dbt tests, SLO engineering.

### Bài tập về nhà

- Hoàn thành checklist capstone Lab #26
- Đọc: Tài liệu Great Expectations checkpoint
- Đọc: Google SRE Workbook — chương SLO engineering